# সহজ ভাষায় Notebook Guide

এই notebook **সম্পূর্ণ self-contained** — project folder লাগে না। একটা ছোট, বানানো clinic handbook (৪টা document, ৮টা section) দিয়ে privacy-first RAG-এর প্রতিটা ধাপ হাতে-কলমে দেখানো হয়েছে। Technical term English-এ, explanation Bangla-তে।

> ⚠️ Handbook-এর সব লেখা **educational, fictional; not medical advice**। "Riverbend Community Clinic" একটা কাল্পনিক clinic।

## কীভাবে ব্যবহার করবেন?

1. উপর থেকে নিচে run করুন। দরকার শুধু `pandas` — বাকি সব Python standard library।
2. **Section 5 optional:** `chromadb` আর `sentence-transformers` লাগে, আর প্রথমবার model download (~61 MB, একবারই)। Cell-টায় `skip-execution` tag আছে, saved notebook-এ তার output নেই। না চালালেও বাকি notebook pure-Python fallback দিয়ে চলে।
3. Section 8-এর Presidio cell-ও optional (spaCy model download লাগে), একই কারণে output নেই।
4. প্রতিটা experiment-এর আগে অনুমান লিখুন, তারপর output-এর সাথে মেলান।

| Section | কী শিখব |
|---|---|
| 1 | Privacy switch — library import-এর আগে |
| 2 | ছোট clinic KB |
| 3 | Section ধরে chunking আর stable id |
| 4 | Embedding আর distance — pure-Python fallback |
| 5 | Real embedding + ChromaDB (optional) |
| 6 | Abstention threshold calibration |
| 7 | Fallback-এর সীমা: paraphrase আর Bangla প্রশ্ন |
| 8 | Regex PII redactor — Bangladeshi phone সহ |
| 9 | Log-এ কী পৌঁছায়: আগে redact |

## 1. Privacy Switch — Import-এর আগে

Chroma আর Hugging Face library import-এর সময়ই env var পড়ে। তাই telemetry বন্ধের switch **import-এর আগে** বসাতে হয় — Week 14-এ CrewAI-র তিনটা flag-এর মতো। `HF_HUB_OFFLINE=1` দিন প্রথম model download-এর **পরে**: তখন থেকে Hub-এ কোনো call যায় না, আর model না থাকলে নিঃশব্দে download না করে সরাসরি error দেয়।

In [1]:
import os

# Privacy switches BEFORE importing chromadb / huggingface libraries: they read env vars at import time.
os.environ.update({
    "ANONYMIZED_TELEMETRY": "False",      # Chroma product telemetry off
    "HF_HUB_DISABLE_TELEMETRY": "1",      # huggingface_hub usage telemetry off
})
# After the first model download also set HF_HUB_OFFLINE=1 -> no Hub calls at all (a missing model fails loudly).

import hashlib, json, math, re, sys
from collections import Counter

import pandas as pd

print("python", sys.version.split()[0], "| pandas", pd.__version__)
print({k: os.environ[k] for k in ("ANONYMIZED_TELEMETRY", "HF_HUB_DISABLE_TELEMETRY")})

python 3.13.14 | pandas 3.1.0
{'ANONYMIZED_TELEMETRY': 'False', 'HF_HUB_DISABLE_TELEMETRY': '1'}


## 2. ছোট Clinic KB

Project-এর KB ২৫-৪০ পাতার একটা নিজে-লেখা handbook; এখানে তার ছোট সংস্করণ। Real patient data বা internet থেকে কপি করা লেখা কেন না?

1. **কোনো PHI নেই**, license-এর প্রশ্নও নেই — repo public করা যায়।
2. **Ground truth জানা** — কোন প্রশ্নের উত্তর কোন section-এ, আমরা জানি; evaluation set সেটার ওপর দাঁড়ায়।
3. **ইচ্ছাকৃত ফাঁক** — malaria বা diabetes diet KB-তে নেই; Abstention ঠিক কাজ করছে কিনা সেটা দিয়েই পরীক্ষা।

প্রতিটা document Markdown: `#` title, `##` section।

In [2]:
DISCLAIMER = "Educational, fictional content; not medical advice."

KB = {
"clinic-services": """# Clinic Services
## Opening hours and appointments
Riverbend Community Clinic is open Sunday to Thursday, 9:00 to 17:00, and closed on public holidays. Book an appointment at the front desk or by phone. Walk-in patients are seen after booked patients.
## Replacing a lost clinic card
If you lose your clinic card, the front desk issues a new one. Bring a photo ID. Your records stay the same.
""",
"fever-adults": """# Fever in Adults
## Fever self-care at home
A fever is a body temperature of 38°C or higher. Rest, drink plenty of fluids and wear light clothing. Check your temperature twice a day and write it down.
## When to see a clinician for fever
See a clinician if a fever lasts more than three days, or comes with a rash, a stiff neck, confusion or repeated vomiting. In the monsoon season a fever with body aches can be dengue; the clinic offers a dengue test.
""",
"dengue": """# Dengue
## Dengue warning signs
Warning signs of severe dengue often appear when the fever goes down: severe stomach pain, persistent vomiting, bleeding gums or nose, blood in vomit or stool, and extreme tiredness or restlessness. Anyone with these signs needs urgent hospital care.
## Preventing mosquito bites
Remove standing water around the house every week, sleep under a bed net, and wear long sleeves in the early morning and evening, when Aedes mosquitoes bite most.
""",
"dehydration": """# Diarrhoea and Dehydration
## Oral rehydration solution (ORS)
Oral rehydration solution replaces the water and salts lost in diarrhoea. Mix one ORS packet with the exact amount of clean water written on the packet. Make a fresh mix every 24 hours. Keep breastfeeding a baby who has diarrhoea.
## Signs of dehydration
Signs of dehydration are a dry mouth, little or dark urine, dizziness and sunken eyes. In children also watch for crying without tears and unusual sleepiness.
""",
}
print(DISCLAIMER, "|", len(KB), "documents,", sum(md.count("\n## ") for md in KB.values()), "sections")

Educational, fictional content; not medical advice. | 4 documents, 8 sections


## 3. Section ধরে Chunking আর Stable ID

Week 12-এ fixed-size chunk (৩০০-৮০০ character, overlap) দেখেছি। Clinical document-এ সেটা বিপজ্জনক: "কখন clinician দেখাবেন" list-টা মাঝখানে কাটা পড়লে retrieval অর্ধেক list আনে, আর উত্তর পড়ে মনে হয় বাকি লক্ষণগুলোর কোনো গুরুত্ব নেই। তাই নিয়ম: **একটা `##` section = একটা chunk**, আর chunk-এর শুরুতে document title + heading — embedding তখন জানে লেখাটা কোন বিষয়ের।

Id হবে `doc#section-slug` — মানুষ পড়তে পারে, citation-এ দেখানো যায়, evaluation set-এ "expected section" হিসেবে লেখা যায়। লেখার `sha` থাকে metadata-তে, যাতে content বদলালে ধরা পড়ে।

In [3]:
def slug(s):
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")

def chunk_by_section(doc_id, markdown):
    """One '## ' section = one chunk; the text starts with 'Title - Heading' so the embedding knows the topic."""
    title, chunks = "", []
    for block in re.split(r"(?m)^(?=#{1,2} )", markdown):
        head, _, body = block.strip().partition("\n")
        if head.startswith("# "):
            title = head[2:].strip()
        elif head.startswith("## "):
            heading = head[3:].strip()
            text = f"{title} - {heading}\n{body.strip()}"
            chunks.append({"id": f"{doc_id}#{slug(heading)}", "text": text,
                           "meta": {"doc": doc_id, "section": heading,
                                    "sha": hashlib.sha256(text.encode()).hexdigest()[:12]}})
    return chunks

def build(kb):
    return [c for doc_id, md in kb.items() for c in chunk_by_section(doc_id, md)]

CHUNKS = build(KB)
pd.DataFrame([{"id": c["id"], "words": len(c["text"].split()), "sha": c["meta"]["sha"]} for c in CHUNKS])

,id,words,sha
0,clinic-services#opening-hours-and-appointments,40,973b36a3c6da
1,clinic-services#replacing-a-lost-clinic-card,30,54750fde8d88
2,fever-adults#fever-self-care-at-home,37,44a24c145ff1
3,fever-adults#when-to-see-a-clinician-for-fever,52,bdd4268737d6
4,dengue#dengue-warning-signs,44,3645da23b63c
5,dengue#preventing-mosquito-bites,33,13e050430096
6,dehydration#oral-rehydration-solution-ors,48,3ce7e2e6dde8
7,dehydration#signs-of-dehydration,33,9aba06089f46


**তিন রকম id-র পরীক্ষা।** Handbook-এর নতুন version-এ তিনটা জিনিস ঘটল: শুরুতে নতুন section (`Home visits`), dengue section-এ একটা শব্দ বদল, আর শেষ section (`Signs of dehydration`) review-এর জন্য সরানো। কোন scheme-এ পুরোনো citation আর eval label ভেঙে যায়?

In [4]:
kb_v2 = dict(KB)
kb_v2["clinic-services"] = KB["clinic-services"].replace(
    "## Opening hours", "## Home visits\nA nurse can visit patients who cannot travel. Ask at the front desk.\n## Opening hours")
kb_v2["dengue"] = KB["dengue"].replace("extreme tiredness", "unusual tiredness")        # small edit
kb_v2["dehydration"] = KB["dehydration"].split("## Signs of dehydration")[0]            # section retired

old, new = build(KB), build(kb_v2)
schemes = {
    "positional (chunk_0, chunk_1, ...)": lambda cs: {c["meta"]["section"]: f"chunk_{i}" for i, c in enumerate(cs)},
    "content hash": lambda cs: {c["meta"]["section"]: c["meta"]["sha"] for c in cs},
    "doc#section-slug": lambda cs: {c["meta"]["section"]: c["id"] for c in cs},
}
rows = []
for name, ids_of in schemes.items():
    a, b = ids_of(old), ids_of(new)
    kept = [s for s in a if s in b]                         # sections that exist in both versions
    rows.append({"id scheme": name, "same section, new id": sum(a[s] != b[s] for s in kept), "out of": len(kept)})
display(pd.DataFrame(rows))

old_sha = {c["id"]: c["meta"]["sha"] for c in old}
print("stale ids to DELETE from the vector store:", sorted(set(old_sha) - {c["id"] for c in new}))
print("ids whose text changed (re-embed):         ", [c["id"] for c in new if c["id"] in old_sha and old_sha[c["id"]] != c["meta"]["sha"]])

,id scheme,"same section, new id",out of
0,"positional (chunk_0, chunk_1, ...)",7,7
1,content hash,1,7
2,doc#section-slug,0,7


stale ids to DELETE from the vector store: ['dehydration#signs-of-dehydration']
ids whose text changed (re-embed):          ['dengue#dengue-warning-signs']


**পড়ার নিয়ম:** positional id-তে নতুন section ঢোকানো মানে তার পরের সব id সরে যাওয়া — পুরোনো log-এর citation আর eval-এর expected section এখন অন্য লেখা দেখায়। Content hash idempotent (Week 15-এর ingestion), কিন্তু একটা typo fix-এও id বদলায়। Slug id একই থাকে, আর `sha` বলে দেয় কোনটা re-embed করতে হবে।

আর **stale id মুছতেই হবে:** `upsert` নতুন chunk যোগ করে, পুরোনোগুলো মোছে না। সরিয়ে দেওয়া (হয়তো ভুল বা পুরোনো) medical লেখা তাহলে retrieval-এ আসতেই থাকবে।

## 4. Embedding আর Distance — Pure-Python Fallback

Real system-এ embedding model প্রতিটা chunk-কে একটা ভেক্টর বানায় (MiniLM-এ ৩৮৪টা সংখ্যা), আর প্রশ্নের ভেক্টরের সাথে **cosine distance** = 1 − cosine similarity মাপে: 0 মানে একই দিক, 1 মানে কোনো মিল নেই। Week 15-এ Chroma-র default `l2` space ছিল — distance 25-26-এর মতো সংখ্যা, যার কোনো স্বাভাবিক স্কেল নেই। এখানে collection `cosine` space-এ, তাই threshold পড়া সহজ।

এই section-এ model ছাড়াই একই ধারণা: TF-IDF word vector + cosine। এটা semantic না, lexical — "doctor" আর "clinician" এর কাছে আলাদা শব্দ। কিন্তু distance, top-k আর threshold-এর mechanics হুবহু এক, আর কোনো download লাগে না।

In [5]:
STOP = set("a an and are as at be by can do does for from has have how i if in is it me my of on or should "
           "the this to was what when where which who will with you your".split())

def tokens(text):
    out = []
    for w in re.findall(r"[a-z0-9]+", text.lower()):
        if w in STOP:
            continue
        for suffix, min_len in (("ing", 5), ("es", 4), ("s", 3)):   # ponytail: toy stemmer; real embeddings handle morphology
            if len(w) > min_len and w.endswith(suffix) and not w.endswith("ss"):
                w = w[: -len(suffix)]
                break
        out.append(w)
    return out

def cosine(a, b):
    dot = sum(v * b.get(t, 0.0) for t, v in a.items())
    na, nb = math.sqrt(sum(v * v for v in a.values())), math.sqrt(sum(v * v for v in b.values()))
    return dot / (na * nb) if na and nb else 0.0

DF = Counter(t for c in CHUNKS for t in set(tokens(c["text"])))

def vec(text):
    idf = lambda t: math.log((1 + len(CHUNKS)) / (1 + DF[t])) + 1      # rare words weigh more
    return {t: n * idf(t) for t, n in Counter(tokens(text)).items()}

INDEX = [(c, vec(c["text"])) for c in CHUNKS]

def search(question, k=3):
    q = vec(question)
    hits = [{"id": c["id"], "distance": round(1 - cosine(q, v), 3), "text": c["text"]} for c, v in INDEX]
    return sorted(hits, key=lambda h: h["distance"])[:k]

print(tokens("What are the warning signs of severe dengue?"))
pd.DataFrame([{k: h[k] for k in ("id", "distance")} for h in search("What are the warning signs of severe dengue?")])

['warn', 'sign', 'severe', 'dengue']


,id,distance
0,dengue#dengue-warning-signs,0.346
1,dehydration#signs-of-dehydration,0.846
2,fever-adults#when-to-see-a-clinician-for-fever,0.904


## 5. Real Embedding + ChromaDB (optional — model download)

> ⚠️ **এই cell saved notebook-এ চালানো হয়নি** (`skip-execution` tag, output নেই)। প্রথমবার `paraphrase-MiniLM-L3-v2` download করে (~61 MB, একবারই; তারপর offline)। আপনার মেশিনে চালালে output-এর আকার হবে: প্রথমে `max_seq_length 128 | dim 384`, তারপর `8 chunks`, তারপর প্রতিটা প্রশ্নের `(id, distance)` list। Distance-এর সংখ্যা model আর version-এর ওপর নির্ভর করে — নিজে মাপুন, এখানে কোনো সংখ্যা দেওয়া হলো না।

তিনটা জিনিস লক্ষ্য করুন:

1. `Settings(anonymized_telemetry=False)` — env var-এর পাশাপাশি code-এও, যাতে কেউ env ভুলে গেলেও বন্ধ থাকে।
2. `configuration={"hnsw": {"space": "cosine"}}` — collection **তৈরির সময়ই**; পরে space বদলানো যায় না, নতুন collection লাগে।
3. `normalize_embeddings=True` — unit vector; তখন cosine আর dot product একই ক্রম দেয়।

`max_seq_length 128` মানে section-এর প্রথম ~১২৮ token-ই embedding-এ ঢোকে, বাকিটা চুপচাপ কাটা পড়ে। তাই KB-র section ছোট রাখা হয়েছে।

In [ ]:
# Needs: pip install chromadb sentence-transformers   (first run downloads the model once, then works offline)
#
# HIGHLIGHTS: embedding model বাছাই — chosen model + বিকল্প
# | Model                                 | Use Case                  | Advantage                          | Limitation                                       |
# | paraphrase-MiniLM-L3-v2 (chosen)      | ছোট local KB, শুধু CPU      | ~61 MB, 384-dim, দ্রুত, course standard | general English; max 128 token — লম্বা section কাটা পড়ে |
# | all-MiniLM-L6-v2                      | general semantic search    | ভালো quality, max 256 token         | একটু বড় ও ধীর, তবু general domain               |
# | pritamdeka/S-PubMedBert-MS-MARCO      | biomedical retrieval       | PubMed vocabulary, medical synonym | BERT-base (~110M param, 768-dim), CPU-তে ধীর; research লেখার ভাষা |
# | intfloat/multilingual-e5-small        | Bangla + English প্রশ্ন      | এক model-এ দুই ভাষা                  | "query: " / "passage: " prefix লাগে; MiniLM-L3-এর চেয়ে ভারী |
# | Cloud embedding API                   | managed scale              | নিজের compute লাগে না               | প্রতিটা প্রশ্ন বাইরে যায় — privacy-first-এর বিপরীত |
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

st = SentenceTransformer("sentence-transformers/paraphrase-MiniLM-L3-v2")

def embed(texts):
    return st.encode(texts, normalize_embeddings=True).tolist()

print("max_seq_length", st.max_seq_length, "| dim", len(embed(["x"])[0]))

client = chromadb.PersistentClient(path="./chroma_w17_lecture", settings=Settings(anonymized_telemetry=False))
col = client.get_or_create_collection("clinic_kb", configuration={"hnsw": {"space": "cosine"}})
col.upsert(ids=[c["id"] for c in CHUNKS], documents=[c["text"] for c in CHUNKS],
           metadatas=[c["meta"] for c in CHUNKS], embeddings=embed([c["text"] for c in CHUNKS]))

def search_chroma(question, k=3):
    r = col.query(query_embeddings=embed([question]), n_results=k)
    return [{"id": i, "distance": round(d, 3), "text": t}
            for i, d, t in zip(r["ids"][0], r["distances"][0], r["documents"][0])]

print(col.count(), "chunks")
for q in ("When should I see a doctor about a fever?", "My stools are watery, what should I drink?"):
    print(q, "->", [(h["id"], h["distance"]) for h in search_chroma(q)])

## 6. Abstention Threshold Calibration

Abstention-এর নিয়ম: সবচেয়ে কাছের chunk-ও যদি threshold-এর চেয়ে দূরে থাকে, LLM-কে **ডাকাই হবে না** — একটা fixed message যাবে। Threshold অনুমানে বসানো যায় না, মাপতে হয় দুই সেট প্রশ্নে: on-topic, যার উত্তর KB-তে আছে, আর off-topic, যার নেই। Off-topic-এর মধ্যে সবচেয়ে কাজের হলো **near-topic** প্রশ্ন — একই domain, কিন্তু KB-তে নেই (malaria-র চিকিৎসা, clinic-এর parking)। Threshold-এর আসল পরীক্ষা এগুলোই; "capital of France" যেকোনো threshold-এই বাদ পড়ে।

`SEARCH = search` — Section 5 চালালে `SEARCH = search_chroma` করে আবার চালান। পদ্ধতি একই, সংখ্যা আলাদা।

In [6]:
SEARCH = search          # after Section 5: SEARCH = search_chroma

ON_TOPIC = ["What time does the clinic open?", "How do I mix ORS for diarrhoea?",
            "What are the warning signs of severe dengue?", "When should I see a doctor about a fever?",
            "How can I stop mosquitoes biting my family?", "What are the signs of dehydration in children?"]
OFF_TOPIC = ["What is the capital of France?", "How do I reset my email password?",
             "What is the treatment for malaria?", "Does the clinic have car parking?", "What diet is good for diabetes?"]

rows = []
for kind, questions in (("on", ON_TOPIC), ("off", OFF_TOPIC)):
    for q in questions:
        best = SEARCH(q, k=1)[0]
        rows.append({"kind": kind, "question": q, "best_id": best["id"], "best_distance": best["distance"]})
calib = pd.DataFrame(rows).sort_values("best_distance", ignore_index=True)
display(calib)

worst_on = calib.loc[calib["kind"] == "on", "best_distance"].max()
best_off = calib.loc[calib["kind"] == "off", "best_distance"].min()
if worst_on < best_off:
    THRESHOLD = round((worst_on + best_off) / 2, 3)
    print(f"gap: on-topic worst {worst_on} < off-topic best {best_off} -> THRESHOLD = {THRESHOLD} (midpoint)")
else:
    THRESHOLD = round(best_off - 0.01, 3)           # overlap: prefer abstaining over answering off-topic
    print(f"OVERLAP: on-topic worst {worst_on} >= off-topic best {best_off} -> THRESHOLD = {THRESHOLD}; "
          "some on-topic questions will abstain -> fix chunking/KB, don't just move the number")

,kind,question,best_id,best_distance
0,on,What are the warning signs of severe dengue?,dengue#dengue-warning-signs,0.346
1,on,What are the signs of dehydration in children?,dehydration#signs-of-dehydration,0.428
2,on,How do I mix ORS for diarrhoea?,dehydration#oral-rehydration-solution-ors,0.466
3,on,When should I see a doctor about a fever?,fever-adults#when-to-see-a-clinician-for-fever,0.692
4,on,What time does the clinic open?,clinic-services#opening-hours-and-appointments,0.714
5,on,How can I stop mosquitoes biting my family?,dengue#preventing-mosquito-bites,0.742
6,off,Does the clinic have car parking?,clinic-services#replacing-a-lost-clinic-card,0.828
7,off,What is the capital of France?,clinic-services#opening-hours-and-appointments,1.000
8,off,How do I reset my email password?,clinic-services#opening-hours-and-appointments,1.000
9,off,What is the treatment for malaria?,clinic-services#opening-hours-and-appointments,1.000


gap: on-topic worst 0.742 < off-topic best 0.828 -> THRESHOLD = 0.785 (midpoint)


**পড়ার নিয়ম:** সবচেয়ে কাছের off-topic প্রশ্নটা near-topic ("clinic"-এর parking) — "clinic" শব্দ মিলেছে, তাই distance 1-এর নিচে। On-topic-এর সবচেয়ে খারাপটা আর এটার মাঝের ফাঁক সরু; threshold সেই ফাঁকের মাঝখানে। দুটো সতর্কতা:

* **Model, chunking বা KB বদলালে আবার calibrate।** Distance-এর বণ্টন বদলায়; অন্য project থেকে সংখ্যা কপি করা চলে না (Week 15-এ chunk 700-এ কোনো threshold-ই কাজ করেনি)।
* **Overlap হলে healthcare-এ abstain-এর দিকে ঝুঁকুন।** "জানি না" বলা একটা correct outcome; KB-র বাইরের medical উত্তর না।

এবার Abstention-কে একটা function বানাই। LLM call হয় শুধু threshold-এর ভেতরে থাকলে:

In [7]:
ABSTAIN_MESSAGE = ("I don't have information about this in the clinic handbook. "
                   "Please ask a clinician or the front desk.")

def retrieve_or_abstain(question, k=3):
    hits = SEARCH(question, k)
    if not hits or hits[0]["distance"] > THRESHOLD:
        return {"route": "abstained", "answer": ABSTAIN_MESSAGE, "llm_called": False}
    return {"route": "answer_with_llm", "passages": [(h["id"], h["distance"]) for h in hits], "llm_called": True}

for q in ["How do I mix ORS for diarrhoea?", "Does the clinic have car parking?"]:
    print(q, "->", retrieve_or_abstain(q))

How do I mix ORS for diarrhoea? -> {'route': 'answer_with_llm', 'passages': [('dehydration#oral-rehydration-solution-ors', 0.466), ('dehydration#signs-of-dehydration', 0.916), ('clinic-services#opening-hours-and-appointments', 1.0)], 'llm_called': True}
Does the clinic have car parking? -> {'route': 'abstained', 'answer': "I don't have information about this in the clinic handbook. Please ask a clinician or the front desk.", 'llm_called': False}


## 7. Fallback-এর সীমা: Paraphrase আর Bangla প্রশ্ন

দুটো প্রশ্নের উত্তর KB-তে আছে (ORS section), কিন্তু একটাও শব্দ মেলে না। **অনুমান করুন:** fallback কী করবে?

In [8]:
for q in ["My stools are watery, what should I drink?",    # on-topic (ORS), zero shared words
          "ডায়রিয়া হলে কী খাওয়াব?"]:                        # same question in Bangla
    best = search(q, k=1)[0]
    print(f"{q!r:48} -> {best['id']:<45} {best['distance']} -> {'abstain' if best['distance'] > THRESHOLD else 'answer'}")
print("tokens the fallback sees in the Bangla question:", tokens("ডায়রিয়া হলে কী খাওয়াব?"))

'My stools are watery, what should I drink?'     -> fever-adults#fever-self-care-at-home          0.899 -> abstain
'ডায়রিয়া হলে কী খাওয়াব?'                      -> clinic-services#opening-hours-and-appointments 1.0 -> abstain
tokens the fallback sees in the Bangla question: []


KB-তে উত্তর থাকা সত্ত্বেও দুটোই abstain — দোষ threshold-এর না, retrieval-এর। এই ধরনের ভুলের দাম একটা "জানি না", ভুল উত্তর না — Abstention-এর নিরাপদ দিক। Semantic embedding প্রথম প্রশ্নটা ধরার কথা ("watery stools" ≈ diarrhoea)। কিন্তু Bangla প্রশ্ন English-only MiniLM-ও ভালো ধরবে না; লাগবে multilingual model (`multilingual-e5-small`) বা Bangla-য় লেখা KB। তাই calibration আর eval set-এ paraphrase আর Bangla প্রশ্ন রাখুন — নইলে এই ফাঁক production-এ গিয়ে ধরা পড়ে।

## 8. Regex PII Redactor — Bangladeshi Phone সহ

User প্রশ্নের সাথে নিজের পরিচয় দিয়ে ফেলে: "My name is Rahim, 01712345678, MRN ..."। KB-র উত্তর খুঁজতে এগুলোর কোনোটাই লাগে না, কিন্তু log, cache আর trace-এ গেলে সেটা PHI। তাই request ঢোকার সাথে সাথে redact।

চারটা design সিদ্ধান্ত:

* **ক্রম গুরুত্বপূর্ণ** — specific আগে, generic পরে: email → MRN → DOB → phone → লম্বা ID-সংখ্যা → name। উল্টো হলে phone নম্বর `[ID]` হয়ে যেত।
* **Bengali digit আগে ASCII-তে** — `০১৭১২...` আর `01712...` তাহলে একই pattern-এ ধরা পড়ে।
* **Trigger শব্দ রেখে দেওয়া** — "My name is [NAME]" পড়া যায়, model-ও বোঝে এখানে একটা নাম ছিল।
* **Value না, শুধু entity type-এর count ফেরত** — redactor নিজেই যেন leak-এর উৎস না হয়।

In [9]:
BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
SEP = r"[\s-]?"                                                    # one optional space or dash between digits
MONTH = r"(?i:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"
DATE = rf"(?:\d{{1,2}}[/.-]\d{{1,2}}[/.-]\d{{2,4}}|\d{{4}}-\d{{2}}-\d{{2}}|\d{{1,2}}\s+{MONTH}\s+\d{{4}})"

# Order matters: specific patterns first; the generic long-number rule near the end.
PII_PATTERNS = [
    ("EMAIL", r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"),
    ("MRN",   r"(?P<keep>(?i:\bMRN|\bmedical record (?:number|no\.?))[\s:#-]*)[A-Za-z0-9][A-Za-z0-9-]{3,}"),
    ("DOB",   rf"(?P<keep>(?i:\bDOB|\bdate of birth|\bborn(?: on)?)[\s:]*){DATE}"),
    ("PHONE", rf"(?<![\d+])(?:\+?88{SEP})?0{SEP}1[3-9](?:{SEP}\d){{8}}(?!\d)"),   # BD mobile: 01XXXXXXXXX, +880 1XXX...
    ("PHONE", r"(?<![\d+])\+\d{1,3}(?:[\s-]?\d){6,12}(?!\d)"),                      # other international numbers
    ("ID",    r"(?<!\d)\d(?:[\s-]?\d){7,16}(?!\d)"),                                 # NID (10/13/17 digits) and similar runs
    ("NAME",  r"(?P<keep>(?i:\bmy name is|\bthis is|\bi am called)\s+)[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,2}"),
    ("NAME",  r"(?P<keep>আমার নাম\s+)[^\s,।]+"),
]
PII_RULES = [(label, re.compile(p)) for label, p in PII_PATTERNS]

def redact(text):
    """Return (redacted_text, {entity_type: count}). Never returns the matched values."""
    text = text.translate(BN_DIGITS)              # ০১৭১২... -> 01712... so one pattern covers both scripts
    found = Counter()
    for label, rx in PII_RULES:
        def replace(m, label=label):
            found[label] += 1
            return (m.groupdict().get("keep") or "") + f"[{label}]"
        text = rx.sub(replace, text)
    return text, dict(found)

redact("My name is Rahim Uddin, phone +880 1712-345678, MRN: RCC-2024-00731")

('My name is [NAME], phone [PHONE], MRN: [MRN]',
 {'MRN': 1, 'PHONE': 1, 'NAME': 1})

এবার একটা test table — শুধু সফল case না, ইচ্ছা করে কয়েকটা **false negative** (PII থেকে গেল) আর **false positive** (PII না, তবু মুছল) রাখা হয়েছে। Expected column আমাদের policy: appointment-এর তারিখ রাখা হয়, শুধু DOB মোছা হয়।

In [10]:
PII_TESTS = [   # (text, expected entity types)
    ("My name is Rahim Uddin and my phone is +8801712345678", {"NAME", "PHONE"}),
    ("call me on 01712-345678 tomorrow",                       {"PHONE"}),
    ("number: +880 1912 345678",                               {"PHONE"}),
    ("017 1234 5678 is my wife's number",                      {"PHONE"}),
    ("আমার নাম করিম, ফোন ০১৮১২৩৪৫৬৭৮",                          {"NAME", "PHONE"}),
    ("email rahim.uddin@example.com",                          {"EMAIL"}),
    ("MRN: RCC-2024-00731 needs a refill",                     {"MRN"}),
    ("DOB 12/03/1990, female",                                 {"DOB"}),
    ("I was born on 5 March 1988",                             {"DOB"}),
    ("NID 1987654321098",                                      {"ID"}),
    ("Dhaka office +1 415 555 0100",                           {"PHONE"}),
    ("I took 500 mg twice today",                              set()),
    ("Clinic opens at 9:00, BP was 120/80",                    set()),
    ("My appointment is on 12/03/2026",                        set()),
    ("I am diabetic and 45 years old",                         set()),
    ("my name is rahim",                                       {"NAME"}),
    ("Please call my son Karim",                               {"NAME"}),
    ("This is Urgent, please reply",                           set()),
    ("0 1 7 1 2 3 4 5 6 7 8",                                  {"PHONE"}),
]
rows = []
for text, expected in PII_TESTS:
    redacted, found = redact(text)
    got, parts = set(found), []
    if expected - got:
        parts.append("FN " + ",".join(sorted(expected - got)))
    if got - expected:
        parts.append("FP " + ",".join(sorted(got - expected)))
    rows.append({"verdict": " / ".join(parts) or "ok", "input": text, "redacted": redacted})
pii_table = pd.DataFrame(rows)
display(pii_table)
print(pii_table["verdict"].value_counts().to_dict())

,verdict,input,redacted
0,ok,My name is Rahim Uddin and my phone is +880171...,My name is [NAME] and my phone is [PHONE]
1,ok,call me on 01712-345678 tomorrow,call me on [PHONE] tomorrow
2,ok,number: +880 1912 345678,number: [PHONE]
3,ok,017 1234 5678 is my wife's number,[PHONE] is my wife's number
4,ok,"আমার নাম করিম, ফোন ০১৮১২৩৪৫৬৭৮","আমার নাম [NAME], ফোন [PHONE]"
5,ok,email rahim.uddin@example.com,email [EMAIL]
6,ok,MRN: RCC-2024-00731 needs a refill,MRN: [MRN] needs a refill
7,ok,"DOB 12/03/1990, female","DOB [DOB], female"
8,ok,I was born on 5 March 1988,I was born on [DOB]
9,ok,NID 1987654321098,NID [ID]


{'ok': 15, 'FN NAME': 2, 'FP NAME': 1, 'FN PHONE / FP ID': 1}


**পড়ার নিয়ম:**

* **FN — ছোট হাতের নাম, trigger ছাড়া নাম** ("my son Karim")। Regex শুধু আকার চেনে, অর্থ না।
* **FP — "This is Urgent"**: `this is` + বড় হাতের শব্দ = নাম ধরে নিয়েছে। Privacy-র জন্য সস্তা ভুল, কিন্তু প্রশ্নের অর্থ নষ্ট হতে পারে।
* **ফাঁক দেওয়া সংখ্যা** (`0 1 7 1 ...`) phone হিসেবে না, `[ID]` হিসেবে মুছেছে — label ভুল, কিন্তু value লুকানো; privacy-র দিক থেকে যথেষ্ট।
* **Appointment-এর তারিখ রাখা হয়েছে** — এটা একটা সচেতন policy। HIPAA Safe Harbor ব্যক্তির সাথে যুক্ত সব তারিখকে (বছর বাদে) identifier ধরে; dataset share করার আগে সেই নিয়ম লাগবে।

পরের ধাপ NER: Microsoft Presidio regex + spaCy NER + context word মেশায়। নিচের cell optional — **saved notebook-এ চালানো হয়নি** (spaCy `en_core_web_lg` download ~৪০০ MB)। চালানোর আগে অনুমান করুন: "my son Karim" কি এবার ধরা পড়বে? ছোট হাতের "rahim"? Bangla নাম?

In [ ]:
# Optional: pip install presidio-analyzer presidio-anonymizer && python -m spacy download en_core_web_lg   (~400 MB, once)
from presidio_analyzer import AnalyzerEngine, Pattern, PatternRecognizer
from presidio_anonymizer import AnonymizerEngine

bd_phone = PatternRecognizer(supported_entity="BD_PHONE",
                             patterns=[Pattern("bd_mobile", PII_PATTERNS[3][1], 0.7)],
                             context=["phone", "mobile", "call", "number"])      # nearby words raise the score
analyzer = AnalyzerEngine()            # default: spaCy NER + built-in recognizers (EMAIL_ADDRESS, PHONE_NUMBER, PERSON, ...)
analyzer.registry.add_recognizer(bd_phone)
anonymizer = AnonymizerEngine()
for text in ["Please call my son Karim on 01712345678", "my name is rahim", "This is Urgent, please reply",
             "আমার নাম করিম"]:
    results = analyzer.analyze(text=text, language="en")
    print(f"{text!r:45} -> {anonymizer.anonymize(text=text, analyzer_results=results).text}")

## 9. Log-এ কী পৌঁছায়: আগে Redact

Redaction-এর জায়গা request-এর **প্রথম লাইন** — তার পরে log, cache, red-flag rule, retrieval, LLM সবাই শুধু redacted লেখা দেখে। নিচে একটা ছোট handler, আর তার log। খেয়াল করুন redaction retrieval-এর ক্ষতি করে কিনা।

In [11]:
LOG = []

def handle(raw_question):
    question, pii = redact(raw_question)          # FIRST line: nothing downstream sees the raw text
    hits = SEARCH(question, k=3)
    route = "abstained" if hits[0]["distance"] > THRESHOLD else "answered"
    LOG.append(json.dumps({"q": question, "pii": pii, "route": route, "top": hits[0]["id"]}, ensure_ascii=False))
    return route

raw = "My name is Rahim Uddin, 01712345678. How do I mix ORS for my son's diarrhoea?"
handle(raw)
handle("আমার নাম করিম, ফোন ০১৮১২৩৪৫৬৭৮। Does the clinic have car parking?")
print("\n".join(LOG))
print("same top chunk with and without PII:", search(raw, 1)[0]["id"] == search(redact(raw)[0], 1)[0]["id"])

{"q": "My name is [NAME], [PHONE]. How do I mix ORS for my son's diarrhoea?", "pii": {"PHONE": 1, "NAME": 1}, "route": "answered", "top": "dehydration#oral-rehydration-solution-ors"}
{"q": "আমার নাম [NAME], ফোন [PHONE]। Does the clinic have car parking?", "pii": {"PHONE": 1, "NAME": 1}, "route": "abstained", "top": "clinic-services#opening-hours-and-appointments"}
same top chunk with and without PII: True


Log-এ নাম বা নম্বর নেই, শুধু entity type-এর count — কতজন user PII লিখছে সেটা মাপা যায়, কে লিখছে সেটা না। প্রথম প্রশ্নে redaction-এর আগে-পরে একই chunk এসেছে: নাম-নম্বর KB খুঁজতে কাজে লাগে না।

কিন্তু দ্বিতীয় log-এর `top` দেখুন: Section 6-এ parking প্রশ্নের সবচেয়ে কাছের chunk ছিল `replacing-a-lost-clinic-card`, এখন `opening-hours-and-appointments`। কারণ `[PHONE]` placeholder retriever-এর কাছে একটা শব্দ — "phone" — আর opening hours section-এ "by phone" লেখা আছে। Route তবু abstained, কিন্তু শিক্ষাটা জরুরি: **placeholder-ও text**। খোঁজার আগে placeholder বাদ দেওয়া যায়, আর eval set-এ PII-সহ প্রশ্ন রাখতেই হয়।

## 📝 Key Takeaways

* **Privacy switch import-এর আগে।** `ANONYMIZED_TELEMETRY=False`, `HF_HUB_DISABLE_TELEMETRY=1`, download-এর পরে `HF_HUB_OFFLINE=1`; code-এও `Settings(anonymized_telemetry=False)`।
* **Section = chunk, id = `doc#section-slug`, `sha` metadata-তে।** Positional id insert-এ ভাঙে, content hash edit-এ; stale id নিজে মুছতে হয়।
* **Cosine space collection তৈরির সময়।** Normalized embedding, distance 0-1-এর কাছাকাছি স্কেলে, threshold পড়া সহজ।
* **Threshold মাপা হয়, বসানো হয় না।** Near-topic প্রশ্নই আসল পরীক্ষা; model/chunking/KB বদলালে আবার calibrate; overlap হলে abstain-এর দিকে।
* **Paraphrase আর Bangla প্রশ্ন eval-এ রাখুন।** Lexical fallback দুটোতেই অন্ধ; English-only embedding Bangla-য় দুর্বল।
* **PII redaction request-এর প্রথম লাইনে।** Specific → generic ক্রম, Bengali digit normalize, count রাখুন value না; regex-এর FN/FP মেপে লিখে রাখুন।

## 🧠 Brain Teasers

1. Section 3-এ `Home visits` section-টা clinic-services-এর **শেষে** যোগ করুন। Positional id-তে কয়টা বদলায়? কেন এটা নিরাপত্তার মিথ্যা অনুভূতি দেয়?
2. Section 6-এ "Is there a pharmacy inside the clinic?" off-topic list-এ যোগ করুন। Threshold কী হয়? কোন on-topic প্রশ্ন এখন বিপদে?
3. Section 5 চালিয়ে `SEARCH = search_chroma` দিয়ে Section 6-7 আবার চালান। Gap বড় হয় না ছোট? "My stools are watery" কি এবার answer হয়?
4. Section 8-এ একটা pattern যোগ করুন যাতে "my name is rahim" ধরা পড়ে, কিন্তু "my name is not important" না। কয়টা নতুন FP আসে?
5. `+৮৮০ ১৭১২-৩৪৫৬৭৮` (Bangla digit, international format) কি redact হয়? না হলে কোন লাইন বদলাবেন?